# **Mount my Drive**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive




---



# **Script Start**

In [3]:
import numpy as np
import json
import os
import csv #[Hyosun]
import pandas as pd #[Hyosun]
import random       #[Hyosun]

# [Hyosun] confusion matrix
import seaborn as sn
import pandas as pd
import matplotlib.pyplot as plt
# [/Hyosun] confusion matrix

In [4]:
def get_immediate_files(a_dir):
    return [name for name in os.listdir(a_dir) if os.path.isfile(os.path.join(a_dir, name))]

# **Emotions**

In [5]:
# dataset downloaded from https://zenodo.org/record/3819968
# please change it to your Emotions dataset path
# [Hyosun] current path: ./src/prep_data/Emotions [/Hyosun]

#[Hyosun] use 7 labels
emotions_path = './data/Emotions_audio_16k/'
labels_arr = ['Angry', 'Happy', 'Sad', 'Neutral', 'Fearful', 'Disgusted', 'Surprised']

In [6]:
cwd = os.getcwd()
print(cwd)

/content


In [14]:
os.chdir('/content/drive/MyDrive/hssast/src/')
cur_dir = os.getcwd()
print(cur_dir)

/content/drive/MyDrive/hssast/src


### **The BEST Performance**

In [8]:
finetune_dir = '/content/drive/MyDrive/hssast/src/finetune'
dataset_folder = '/Emotions'

In [9]:
result_dir = '2024-01-16/15:03:47PM-test01-emotions-comp_fusion-True-comp_fusion_method-use_all_patch-comp_fusion_multi_layer-[4,11]-pooling-max-comp_fusion_mlp6-loss-CE-f10-16-t10-16-b48-lr1e-4-ft_avgtok-base--SSAST-Base-Patch-400-1x-noiseTrue'

In [ ]:
exp_dir = finetune_dir + dataset_folder + '/exp/'+ result_dir+'/'
print(exp_dir)

In [ ]:
# os.chdir(cur_dir+dataset_folder+'/exp/'+ result_dir+'/')

In [11]:
# cur_dir = os.getcwd()
print(cur_dir)

/content/drive/MyDrive/hssast/src/models


### **Load the saved best pretrained model**






In [12]:
!pip install timm==0.4.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.4/287.4 kB 4.3 MB/s eta 0:00:00


In [15]:
from models import ASTModel

In [ ]:
# import sys
# model_dir = '/content/drive/MyDrive/hssast/src/models'
# sys.path.insert(1, model_dir)
# from models import ASTModel

In [20]:
# fine-tuning stage
# now you have a labeled dataset you want to finetune AST on
# suppose the avg length is 100 frames (1s) and there are 7 classes
# the fshape and tshape must be same in pretraining and finetuning
# but fstride and tstride can be different in pretraining and finetuning
# using smaller strides improves the performance but also increase the computational overhead
# set pretrain_stage as False since now is in the finetuning stage
# provide the path of the pretrained model you want to load
# ast_mdl = torch.nn.DataParallel(ast_mdl)
input_tdim = 512 #[Hyosun] 512 #100  # fine-tuning data length can be different with pretraining data length
ast_mdl = ASTModel( comp_fusion='True',    #[Hyosun] comp_fusion added
                    comp_fusion_method='use_all_patch',        #[Hyosun] comp_fusion_method added
                    comp_fusion_multi_layer='[4,11]',   #[Hyosun] comp_fusion_multi_layer added
                    pooling_ty='max',                #[Hyosun] pooling_ty added
                    mlp_layers=6,                #[Hyosun] mlp_layers added #[Hyosun] representation shell만 만들어
      # #[Hyosun] comp_fusion logic added
      # comp_fusion='True'
      # comp_fusion_method='use_all_patch'
      # comp_fusion_multi_layer='[4,11]'
      # pooling_ty='max_max' #[Hyosun] #choices=["mean", "min", "max", "mean_min", "mean_max"] #[/Hyosun]
      # mlp_layers=2
      # #[/Hyosun] comp_fusion logic added
                    label_dim=7, #[Hyosun]label_dim=7
                    fshape=16, tshape=16, fstride=10, tstride=10,
                    #[Hyosun]fshape=16, tshape=16, fstride=10, tstride=10 meaning we allow overlap for 6 pixels
                    ### From the paper SSAST: In the fine-tuning and inference steps,
                    ### we split the patch with an overlap of 6 in the same fashion as the original AST. [/Hyosun]
                    input_fdim=128, input_tdim=input_tdim, model_size='base',
                    pretrain_stage=False, load_pretrained_mdl_path= exp_dir + 'fold1/models/best_audio_model.pth')
# # alternatively, use a frame based AST model
# ast_mdl = ASTModel(label_dim=35,
#              fshape=128, tshape=2, fstride=128, tstride=1,
#              input_fdim=128, input_tdim=input_tdim, model_size='base',
#              pretrain_stage=False, load_pretrained_mdl_path='./test_mdl.pth')

# do finetuning, see src/traintest.py for our finetuning code

# [Hyosun] change here for our dataset CREMA-D
# test_input = torch.zeros([10, input_tdim, 128])
# test_input =

# [Hyosun] prediction
# prediction = ast_mdl(test_input, task='ft_avgtok')

# output should in shape [batch_size, label_dim]
# print(prediction.shape)

# calculate the loss, do back propagate, etc

ValueError: The model loaded is not from a torch.nn.Dataparallel object. Wrap it with torch.nn.Dataparallel and try again.

### **create the label description dictionary**

In [ ]:
# [Hyosun: create the label description dictionary]
if os.path.exists('./../../../data/emotions_class_labels_indices.csv') == True:
    label_set = np.loadtxt('./../../../data/emotions_class_labels_indices.csv', delimiter=',', dtype='str')
    label_map = {}
    for i in range(1, len(label_set)):
        print("[Hyosun] label_set[i][0]: ", label_set[i][0])
        print("[Hyosun] label_set[i][2]: ", label_set[i][2])
        label_map[(label_set[i][2])] = label_set[i][0]
    print(label_map)
    print("[Hyosun] create the label description dictionary: using emotions_class_labels_indices.csv finished!!")
#[/Hyosun: create the label description dictionary]

### **Load target(test_labels)**

In [ ]:
# read target.csv vs predictions
if os.path.exists('./../../../data/evaluation_setup/fold1_test.csv') == True:

    # [Hyosun] load target, i.e., test_labels
    test_meta = np.loadtxt('./../../../data/evaluation_setup/fold1_test.csv', delimiter='\t', dtype='str', skiprows=1)
    print(np.shape(test_meta))
    test_labels = []
    for i in range(0, len(test_meta)):
        # print("test_meta[i][1]: ", test_meta[i][1])
        cur_label = label_map[test_meta[i][1]]
        test_labels.append(int(cur_label))

    print("[Hyosun] test_labels[500]: ", test_labels[500])
    print("[Hyosun] test_labels information load: finished!!")
    #[/Hyosun] load target, i.e., test_labels

### **Load Predictions**

In [ ]:
import csv
if os.path.exists('./result.csv') == True:
    # [Hyosun] get the best_epoch
    results = np.loadtxt('./result.csv', delimiter=',', dtype='str')
    # rows = []
    # with open('./result.csv', 'r') as file:
    #     csvreader = csv.reader(file)
    #     for row in csvreader:
    #         rows.append(row[0])
    # print(rows)
    print("np.shape(results): ", np.shape(results))
    # print(results)
    print("results[:,0]: \n", results[:,0])
    print("results[:,0].shape: \n", results[:,0].shape)
    best_epoch_1 = np.argmax(results[:,0]) #[Hyosun] +1 needed for epoch count later & Only the first occurrence is returned.
    print("best_epoch_1: ", best_epoch_1)
    print("\n")

    # [Hyosun] load predictions
    pred_dir = cur_dir+'/fold1/predictions/predictions_'+str(best_epoch_1+1)+'.csv' #[Hyosun] +1 needed for epoch count
    if os.path.exists(pred_dir) == True:
        pred = np.loadtxt(pred_dir, delimiter=',', dtype='str')
        print("pred.shape[0]: ", pred.shape[0])
        print("pred.shape: ", pred.shape)
        pred_labels = []
        for i in range(0, pred.shape[0]):
            pred_label = np.argmax(pred[i])
            # print("test_meta[i][1]: ", test_meta[i][1])
            pred_labels.append(pred_label)
        print("size of pred_labels: ", np.size(pred_labels))
        print("pred_labels[700:799]: ", pred_labels[700:799])

In [ ]:
np.size(pred_labels)

In [ ]:
np.size(test_labels)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
# print(pred_labels)
cm = confusion_matrix(test_labels,pred_labels)

df_cm = pd.DataFrame(cm, index = [i for i in labels_arr],
                columns = [i for i in labels_arr])

# fig = plt.figure()
fig = plt.figure(figsize = (12,10))
sn.heatmap(df_cm, annot=True, fmt='g') #[Hyosun] fmt='g' added to remove e from the count
plt.xticks(rotation=45)
plt.yticks(rotation=45)
plt.xlabel('Predictions')
plt.ylabel('True Labels')
plt.title('Confusion Matrix'+result_dir, fontdict={'fontsize': 5}, loc='center', )
# fig.savefig('confusion_matrix.jpg')

fig.savefig('Emotions_conf_mat.png',dpi=300)



In [ ]:
best_epoch_1

In [ ]:
best_epoch_1+1 #[Hyosun] real best epoch

In [ ]:
results[best_epoch_1,0] #[Hyosun] real best acc

In [ ]:
        # if main_metrics == 'mAP':
        #     result[epoch-1, :] = [mAP, mAUC, average_precision, average_recall, d_prime(mAUC), loss_meter.avg, valid_loss, cum_mAP, cum_mAUC, optimizer.param_groups[0]['lr']]
        # else:
        #     result[epoch-1, :] = [acc, mAUC, average_precision, average_recall, d_prime(mAUC), loss_meter.avg, valid_loss, cum_acc, cum_mAUC, optimizer.param_groups[0]['lr']]
        # np.savetxt(exp_dir + '/result.csv', result, delimiter=',')
        # print('validation finished')

- result.csv에 들어가는 10 column들 순서:
    - [acc, mAUC, average_precision, average_recall, d_prime(mAUC), loss_meter.avg, valid_loss, cum_acc, cum_mAUC, optimizer.param_groups[0]['lr']]
